<!-- -*- mode: markdown; coding: utf-8; fill-column: 60; ispell-dictionary: "english" -*- -->

<meta charset="utf-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1"/>
<link rel="stylesheet" href="style.css">


# EDAF75 - lab 2: Testing the database

As usual we have to tell Jupyter to allow SQL:

In [1]:
%load_ext sql

And then we import our movie database

In [2]:
%sql sqlite:///movies.sqlite

We want to make sure that SQLite3 really checks our foreign
key constraints -- to do that, we run:

In [3]:
%%sql
PRAGMA foreign_keys=ON;

 * sqlite:///movies.sqlite
Done.


[]

Now write SQL code for the following tasks:


+ Show the names of all movies.

In [4]:
%%sql
SELECT movie_title, movie_year
FROM movie

 * sqlite:///movies.sqlite
Done.


movie_title,movie_year
Inception,2010
Interstellar,2014
The Grand Budapest Hotel,2014
The Rise and Fall of Scooby Doo,2002
The Rise and Fall of Scooby Doo,2019
Titanic,1997


+ Show the performance dates for one of the movies.

In [5]:
%%sql
SELECT movie_title, start_date, start_time, theater_name
FROM   screening
WHERE  movie_title = 'Titanic' AND movie_year = 1997

 * sqlite:///movies.sqlite
Done.


movie_title,start_date,start_time,theater_name
Titanic,2024-02-10,14:00,Filmstaden
Titanic,2024-02-10,18:00,Filmstaden
Titanic,2024-03-08,18:00,Filmstaden
Titanic,2024-03-09,19:30,Kino


+ Show all data concerning performances at a given theatere
  on a given date.

In [6]:
%%sql
SELECT theater_name, start_date, movie_title, movie_year, start_time
FROM   screening
WHERE  theater_name = 'Filmstaden' AND start_date = '2024-02-10'

 * sqlite:///movies.sqlite
Done.


theater_name,start_date,movie_title,movie_year,start_time
Filmstaden,2024-02-10,Titanic,1997,14:00
Filmstaden,2024-02-10,Titanic,1997,18:00


+ List all customers

In [7]:
%%sql
SELECT *
FROM customer

 * sqlite:///movies.sqlite
Done.


username,full_name,pass_wrd
vitooo,Victor Truong,1234
freddy,Fredrik Orheim,2345
Bona,Jona Waldfogel,3456
john_doe,John Doe,password123
jane_smith,Jane Smith,securepassword
alex_brown,Alex Brown,anotherpass


+ List all tickets

In [8]:
%%sql
SELECT *
FROM ticket

 * sqlite:///movies.sqlite
Done.


ticket_id,customer_username,theater_name,movie_title,movie_year,start_date,start_time


+ Create a new ticket to some performance (i.e., insert a
  new row in your table of tickets).

In [9]:
%%sql
INSERT
INTO   ticket(customer_username, theater_name, movie_title, movie_year, start_date, start_time)
VALUES ('vitooo', 'Filmstaden', 'Interstellar', 2014, '2024-02-13', '19:00'),
       ('freddy', 'Kino',       'Inception',    2010, '2024-02-14', '21:00'),
       ('Bona',   'Kino',       'The Rise and Fall of Scooby Doo', 2002, '2024-02-11', '20:00');


 * sqlite:///movies.sqlite
3 rows affected.


[]

In newer versions of SQLite (since version 3.35, released
  in March 2021), and in
  [PostgreSQL](https://www.postgresql.org/docs/current/sql-insert.html),
  we can get any value generated during an insert using the
  `INSERT...-RETURNING` statement:

~~~{.sql}
INSERT
INTO       students
VALUES     ('Amy', 3.9, 1200)
RETURNING  s_id
~~~


which would return the generated `s_id` for the new
  student.

  If your SQLite version is older than 3.35, and you can't
  upgrade, you can instead use the following idea: each row
  in a SQLite3 table has a `rowid` attribute, it is a unique
  integer which essentially tells in which order the rows
  were inserted, and it's not displayed in queries unless we
  ask for it. SQLite3 also have a function,
  `last_insert_rowid()`, which returns the `rowid` of the
  last inserted row of a table, so we can see the `s_id` of
  the most recently inserted student with the following
  query:

~~~{.sql}
SELECT s_id
FROM   students
WHERE  rowid = last_insert_rowid();
~~~


Now, check what ticket number we got for the ticket we
  created above (it should be the same as the ticket id,
  which should be a `randomblob`):

In [10]:
%%sql
SELECT ticket_id, customer_username, movie_title, movie_year
FROM ticket


 * sqlite:///movies.sqlite
Done.


ticket_id,customer_username,movie_title,movie_year
9f7de58ed743df645c2b203bec0bb328,vitooo,Interstellar,2014
b5211ba2240af7850ebdd28f5255d989,freddy,Inception,2010
b4b42400a071c28a1dc6b98f6c44683e,Bona,The Rise and Fall of Scooby Doo,2002


+ Try to insert two movie theaters with the same name (this
  should fail).

In [11]:
%%sql
INSERT
INTO theater(theater_name, capacity)
VALUES ('Filmstaden', 2000)

 * sqlite:///movies.sqlite
(sqlite3.IntegrityError) UNIQUE constraint failed: theater.theater_name
[SQL: INSERT
INTO theater(theater_name, capacity)
VALUES ('Filmstaden', 2000)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


+ Try to insert a performance where the theater doesnâ€™t
  exist in the database (this should fail).

In [12]:
%%sql
INSERT
INTO   screening(start_date, start_time, theater_name, movie_title, movie_year)
VALUES ('2024-02-10', '18:00', 'Cinema', 'Titanic', 1997)

 * sqlite:///movies.sqlite
(sqlite3.IntegrityError) FOREIGN KEY constraint failed
[SQL: INSERT
INTO   screening(start_date, start_time, theater_name, movie_title, movie_year)
VALUES ('2024-02-10', '18:00', 'Cinema', 'Titanic', 1997)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


+ Create a ticket where either the user or the performance
  doesnâ€™t exist (this should fail).

In [13]:
%%sql
INSERT
INTO   ticket(customer_username, theater_name, movie_title, movie_year, start_date, start_time)
VALUES ('Bona', 'Kino', 'The Revenant', 2018, '2024-02-11', '20:00')


 * sqlite:///movies.sqlite
(sqlite3.IntegrityError) FOREIGN KEY constraint failed
[SQL: INSERT
INTO   ticket(customer_username, theater_name, movie_title, movie_year, start_date, start_time)
VALUES ('Bona', 'Kino', 'The Revenant', 2018, '2024-02-11', '20:00')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [14]:
%%sql
INSERT
INTO   ticket(customer_username, theater_name, movie_title, movie_year, start_date, start_time)
VALUES ('wtff_dude', 'Filmstaden', 'Titanic', 1997, '2024-02-10', '18:00')


 * sqlite:///movies.sqlite
(sqlite3.IntegrityError) FOREIGN KEY constraint failed
[SQL: INSERT
INTO   ticket(customer_username, theater_name, movie_title, movie_year, start_date, start_time)
VALUES ('wtff_dude', 'Filmstaden', 'Titanic', 1997, '2024-02-10', '18:00')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)
